# Modules over vertex algebras

A (left) module over a vertex (super)algebra $V$ is a space $M$ with fields $Y_M(a, z) = \sum_n a_{(n)} z^{-n-1}$ for $a \in V$, such that $a_{(n)} m = 0$ for $n$ large, the vacuum acts as the identity and the Borcherds identity holds. The library has

* the category `VertexAlgebraModules(V)` of modules over $V$ (with super and conformally graded versions; `TestSuite` checks the module axioms),
* a catch-all base class `VertexAlgebraModule`, and constructors for specific types:
  * `V.adjoint_module()`: $V$ acting on itself;
  * `V.simple_module(λ)`, `V.simple_modules()`: the simple modules $V_{\lambda+L}$ of a lattice vertex algebra $V_L$, $\lambda \in L^\circ/L$;
  * `A.module(matrices)`: finite-dimensional modules over a Zhu algebra $A$ (the top levels of positive energy modules);
  * `V.induced_module(U)`, `V.verma_module(h)`: Zhu's universal positive energy module $\bar M(U)$ with top level $U$ (Verma modules, Weyl modules);
  * `M.submodule(vectors)`, `M / N`, `M.maximal_submodule()`, `M.simple_quotient()` (Zhu's $L(U)$), `M.direct_sum(N)`;
  * `Q.induced_module_up_to(U, max_weight, max_depth)`: Dong–Li–Mason's universal module $\bar M_Q(U)$ of a quotient $Q = V/I$ (also of a presentation of a subalgebra), up to a depth.

The elements of $V$ act on module vectors through their own methods, with the conventions of the vertex algebra: `a.nproduct(m, n)` $= a_{(n)} m$, `a.ope(m)` (entry $k$ is $a_{(k)} m$), `a.no(m)` $=$ `a * m` $= a_{(-1)} m$, and `a.mode(m, n)` $= a_n m$ in the physics convention ($a_n = a_{(n + \Delta_a - 1)}$).

Contents:

1. the category and the adjoint module;
2. lattice modules $V_{\lambda+L}$;
3. modules over Zhu algebras;
4. induced modules: Verma modules of the Virasoro and $W_3$ algebras, Weyl modules of affine $\mathfrak{sl}_2$, singular vectors;
5. submodules, quotients, simple modules (minimal models, Lee–Yang, affine $\mathfrak{sl}_2$ at levels 1 and 2), direct sums;
6. gradings by abelian groups (lattice modules, weights of affine modules, a fractional $N = 2$ charge);
7. infinite top levels: affine Verma modules, admissible levels;
8. modules over quotients (Lee–Yang; the triplet algebra $\mathcal W(2)$ is in `triplet_modules.ipynb`);
9. what is not there yet.

In [1]:
from vertex_algebras import (UniversalVertexAlgebra, NormallyOrderedExpressions, LatticeVertexAlgebra,
                             UniversalAffineVertexAlgebra)
from vertex_algebras.vertex_algebra_categories import VertexAlgebraModules
from vertex_algebras.zhu_algebra import LatticeZhuAlgebra

## 1. The category and the adjoint module

Modules over $V$ form a category over $V$ (as Sage's modules over an algebra), contained in the modules over the base ring:

In [2]:
K.<c> = Frac(QQ['c'])
E.<L> = NormallyOrderedExpressions(K)
Vir = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2]}, conformal_vector=L)
C = VertexAlgebraModules(Vir)
print(C)
print(C.Super().Graded().WithBasis())
print(C.Super().Graded().WithBasis().is_subcategory(Modules(K).WithBasis()))

Category of modules over Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field
Category of conformally graded super modules with basis over Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field
True


$V$ is a module over itself. Its vectors are written `M(v)` for $v \in V$, and the vacuum prints as `|0>`:

In [3]:
M = Vir.adjoint_module()
print(M, '\n', M.category())
L = Vir.gen('L')
v = M(L.no(L))
print(L.ope(v))
print(L.mode(v, 0) == 4*v, L * v == M(L.no(L.no(L))), v.degree())
print(M.character(8) == Vir.character(8))
TestSuite(M).run()

Adjoint module of Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field 
 Category of conformally graded modules with basis over Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field
[2*:dL L: + 1/6*d^3L, 4*:L L:, 3*dL, (c+8)*L, 0, 3*c*|0>]
True True 4
True


## 2. Lattice modules $V_{\lambda+L}$

For a lattice vertex algebra $V_L$ and $\lambda$ in the dual lattice $L^\circ = \{\mu : B(\mu, L) \subset \mathbb Z\}$, the module $V_{\lambda+L}$ is the Fock space tensored with the $e^p$, $p \in \lambda + L$:
$$Y(e^q, z)\, e^p = \varepsilon(q, p - \lambda)\, z^{B(q,p)}\, E^-(q, z) E^+(q, z)\, e^{q+p}.$$
These are the simple modules of $V_L$, one for each coset of $L^\circ/L$, for any non-degenerate lattice. $\lambda$ is given in the coordinates of the basis of $L$; the representative with coordinates in $[0, 1)$ is used.

The $A_1$ lattice ($B = (2)$, $V_L = L_1(\mathfrak{sl}_2)$): two modules. The top level of $V_{1/2+L}$ is the doublet $e^{\pm 1/2}$ of weight $1/4$:

In [4]:
A1 = LatticeVertexAlgebra(QQ, matrix([[2]]))
J = A1.J(0)
V = A1.with_conformal_vector(J.no(J) / 4)
M0, M1 = V.simple_modules()
print(M1)
print(M1.weight_space(1/4).basis())
print(M1.character(5))

Module V_(λ+L), λ = (1/2), over Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field
[e^(-1/2), e^(1/2)]
2*q^(1/4) + 2*q^(5/4) + 6*q^(9/4) + 8*q^(13/4) + 14*q^(17/4) + O(q^5)


The zero modes of $e^{\alpha}$, $J$, $e^{-\alpha}$ (the currents $E$, $H$, $F$ of $\mathfrak{sl}_2$) on the doublet:

In [5]:
top = M1.weight_space(1/4).basis()
Ev, Hv, Fv = V.e((1,)), J, V.e((-1,))
def zero_mode_matrix(a):
    return matrix(QQ, [[a.nproduct(u, 0).monomial_coefficients().get(w.leading_support(), 0)
                        for u in top] for w in top])
Em, Hm, Fm = [zero_mode_matrix(a) for a in (Ev, Hv, Fv)]
print(Em, '\n'); print(Hm, '\n'); print(Fm)
print(Em*Fm - Fm*Em == Hm, Hm*Em - Em*Hm == 2*Em)

[0 0]
[1 0] 

[-1  0]
[ 0  1] 

[0 1]
[0 0]
True True


$A_2$: three modules, of lowest weights $0$, $1/3$, $1/3$ with top levels of dimension $1$, $3$, $3$ (`highest_weight_vector()` is a vector of lowest weight). The top levels are those the lattice Zhu algebra finds:

In [6]:
A2 = LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]))
J0, J1 = A2.heisenberg_currents()
VA2 = A2.with_conformal_vector((2*J0.no(J0) + J0.no(J1) + J1.no(J0) + 2*J1.no(J1)) / 6)
for N in VA2.simple_modules():
    lam = N.coset_representative()
    h = N.highest_weight_vector().degree()
    print(lam, [N.weight_space(h + d).dimension() for d in range(4)], N.weight_space(h).basis())
print(LatticeZhuAlgebra(VA2).top_levels())

(0, 0) [1, 8, 17, 46] [|0>]
(1/3, 2/3) [3, 9, 27, 57] [e^(-2/3,-1/3), e^(1/3,-1/3), e^(1/3,2/3)]
(2/3, 1/3) [3, 9, 27, 57] [e^(-1/3,-2/3), e^(-1/3,1/3), e^(2/3,1/3)]
[(0, [(0, 0)]), (1/3, [(-2/3, -1/3), (1/3, -1/3), (1/3, 2/3)]), (1/3, [(-1/3, -2/3), (-1/3, 1/3), (2/3, 1/3)])]


An odd lattice gives a vertex superalgebra and super modules: $\sqrt 3\,\mathbb Z$ ($B = (3)$, the $N = 2$ algebra at $c = 1$) has three modules, of lowest weights $0$, $1/6$, $1/6$; $e^{\lambda + q}$ has parity $B(q, q)$:

In [7]:
L3 = LatticeVertexAlgebra(QQ, matrix([[3]]))
J3 = L3.J(0)
V3 = L3.with_conformal_vector(J3.no(J3) / 6)
for N in V3.simple_modules():
    v = N.highest_weight_vector()
    print(N.coset_representative(), v, v.degree(), N.character(3))
print(N.category())
N = V3.simple_module([1/3])
print(N.weight_space(2/3).basis(), N.monomial(((), (-2/3,))).parity())

(0,) |0> 0 1 + q + 2*q^(3/2) + 2*q^2 + 2*q^(5/2) + O(q^3)
(1/3,) e^(1/3) 1/6 q^(1/6) + q^(2/3) + q^(7/6) + q^(5/3) + 2*q^(13/6) + 3*q^(8/3) + O(q^3)
(2/3,) e^(-1/3) 1/6 q^(1/6) + q^(2/3) + q^(7/6) + q^(5/3) + 2*q^(13/6) + 3*q^(8/3) + O(q^3)
Category of conformally graded super modules with basis over Lattice vertex algebra of rank 1 with Gram matrix [3] over Rational Field
[e^(-2/3)] 1


A background charge changes the weights, $\Delta(p) = B(p, p)/2 + \beta\cdot p$; here $T = T_{\mathrm{sug}} + \frac12 \partial J$ on $A_1$:

In [8]:
Vb = A1.with_conformal_vector(J.no(J) / 4 + J.derivative() / 2)
print(Vb.central_charge())
for N in Vb.simple_modules():
    print(N.coset_representative(), N.character(3))

-5
(0,) 2 + 2*q + 6*q^2 + O(q^3)
(1/2,) q^(-1/4) + 3*q^(3/4) + 4*q^(7/4) + 7*q^(11/4) + O(q^3)


Indefinite lattices work as well. Their weight spaces are infinite dimensional; with the lattice grading of $V$ (applied to $p - \lambda$) the graded pieces are finite. $B = \begin{pmatrix} 0 & 2 \\ 2 & 0 \end{pmatrix}$ has four modules:

In [9]:
H2 = LatticeVertexAlgebra(QQ, matrix([[0, 2], [2, 0]]))
K0, K1 = H2.heisenberg_currents()
Hg = H2.with_conformal_vector((K0.no(K1) + K1.no(K0)) / 4).with_horizontal_grading('lattice')
Ms = Hg.simple_modules()
print([N.coset_representative() for N in Ms])
N = Ms[1]
print(N.weight_space(1).is_finite_dimensional(), [N.weight_space(w, horizontal_grade=(0, 0)).dimension() for w in range(5)])

[(0, 0), (0, 1/2), (1/2, 0), (1/2, 1/2)]


False [0, 0, 0, 0, 0]


## 3. Modules over Zhu algebras

A finite-dimensional module over a Zhu algebra $A$ is given by the matrices of its generators; they are checked against the defining relations of $A$ (and, for the Zhu algebra of a quotient, against its ideal). By Zhu's correspondence these are the possible top levels of positive energy modules.

The Virasoro algebra: $A = \mathbb C[x]$, $x = [L]$, has a one-dimensional module for every $h$. The Lee–Yang model only for its two weights $0$ and $-1/5$:

In [10]:
EQ.<Lq> = NormallyOrderedExpressions(QQ)
VirLY = UniversalVertexAlgebra(QQ, EQ, {('Lq', 'Lq'): [Lq.derivative(), 2*Lq, 0, -11/5]}, conformal_vector=Lq)
Lq = VirLY.gen('Lq')
Z = VirLY.quotient([Lq.no(Lq) - 3/10*Lq.derivative(2)]).zhu_algebra_up_to(4)
print(Z.module({'Lq': -1/5}))
try:
    Z.module({'Lq': 1})
except ValueError as err:
    print('ValueError:', err)

Module of dimension 1 over Quotient of Zhu algebra of Universal vertex algebra on generators ['Lq'] over Rational Field by the images of the ideal of weight <= 4: an algebra mapping onto the Zhu algebra of Quotient of Universal vertex algebra on generators ['Lq'] over Rational Field by the ideal generated by [:Lq Lq: - 3/10*d^2Lq]
ValueError: the matrices do not satisfy the relation [Lq] + 5*[Lq]^2 = 0 of the Zhu algebra


Affine $\mathfrak{sl}_2$: $A = U(\mathfrak{sl}_2)$. The doublet is a module; at level 1 the Zhu algebra of $L_1(\mathfrak{sl}_2)$ takes the doublet but not the triplet:

In [11]:
VA = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=1, names=['E', 'H', 'F'], conformal_vector='sugawara')
AA = VA.zhu_algebra()
doublet = {'E': matrix([[0, 1], [0, 0]]), 'H': matrix([[1, 0], [0, -1]]), 'F': matrix([[0, 0], [1, 0]])}
triplet = {'E': matrix([[0, -2, 0], [0, 0, 1], [0, 0, 0]]), 'H': matrix([[2, 0, 0], [0, 0, 0], [0, 0, -2]]),
           'F': matrix([[0, 0, 0], [-1, 0, 0], [0, 2, 0]])}
U2 = AA.module(doublet)
print(U2, '\n', U2.representation_matrix(AA(VA.conformal_vector())))
E1 = VA.gen('E')
ZL = VA.quotient([E1.no(E1)]).zhu_algebra_up_to(3)
print(ZL.module(doublet).dimension())
try:
    ZL.module(triplet)
except ValueError as err:
    print('ValueError:', err)

Module of dimension 2 over Zhu algebra of Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field 
 [1/4   0]
[  0 1/4]
2
ValueError: the matrices do not satisfy the relation [F]^2 = 0 of the Zhu algebra


Super modules: the $bc$ system (weights $1, 0$) has the Clifford algebra as its Zhu algebra, with the module $\mathbb C^{1|1}$ ($c$ has weight $0$, so the library only claims an algebra mapping onto the Zhu algebra). The H-twisted (Ramond) Zhu algebra of the $N = 1$ algebra has $[G]^2 = [L] - c/24$; in the Ramond sector of the tricritical Ising model ($c = 7/10$) the relation $1280[L]^2 - 608[L] + 21 = 0$ of the quotient leaves the modules of dimension $1|1$ at $h = 3/80$ and $h = 7/16$:

In [12]:
X.<b, cg> = NormallyOrderedExpressions(QQ, parities={'b': 1, 'cg': 1})
BC = UniversalVertexAlgebra(QQ, X, {('b', 'cg'): [1]}, conformal_vector=-b*cg.derivative())
Ubc = BC.zhu_algebra().module({'b': matrix([[0, 1], [0, 0]]), 'cg': matrix([[0, 0], [1, 0]])}, parities=[0, 1])
print(Ubc, '\n', Ubc.category())

NT.<Lt, Gt> = NormallyOrderedExpressions(QQ, parities={'Gt': 1})
TI = UniversalVertexAlgebra(QQ, NT, {('Lt', 'Lt'): [Lt.derivative(), 2*Lt, 0, 7/20],
                                     ('Lt', 'Gt'): [Gt.derivative(), 3/2*Gt],
                                     ('Gt', 'Gt'): [2*Lt, 0, 7/15]}, conformal_vector=Lt)
Ltv, Gtv = TI.gens()
S4 = TI.weight_space(4).basis()
rows = []
for g, n in [(Ltv, 2), (Ltv, 3), (Gtv, 1), (Gtv, 2)]:          # killed by L_1, L_2, G_{1/2}, G_{3/2}
    imgs = [g.nproduct(s, n).monomial_coefficients() for s in S4]
    ks = sorted(set(kk for d in imgs for kk in d))
    rows += [[d.get(kk, 0) for d in imgs] for kk in ks]
[sv] = [sum(cf*s for cf, s in zip(r, S4)) for r in matrix(QQ, rows).right_kernel_matrix().rows()]
ZR = TI.quotient([sv]).zhu_algebra_up_to(6, twist='ramond')
def ramond(h):
    return {'Lt': matrix([[h, 0], [0, h]]), 'Gt': matrix([[0, 1], [h - 7/240, 0]])}
for h in (3/80, 7/16, 1):
    try:
        ZR.module(ramond(h), parities=[0, 1])
        print(h, 'a module')
    except ValueError as err:
        print(h, 'ValueError:', err)

Module of dimension 2 over Algebra mapping onto the Zhu algebra of Universal vertex algebra on generators ['b', 'cg'] over Rational Field

 
 Category of finite dimensional super modules with basis over Rational Field
3/80 a module
7/16 a module
1 ValueError: the matrices do not satisfy the relation 21 - 608*[Lt] + 1280*[Lt]^2 = 0 of the Zhu algebra


## 4. Induced modules

For a universal vertex algebra with generators of positive weight and a module $U$ over its Zhu algebra, `V.induced_module(U)` is Zhu's universal positive energy module $\bar M(U)$ with top level $U$. Its basis is the ordered monomials of the negative modes of the strong generators on a basis of $U$; positive modes kill $U$, zero modes act by the matrices of $U$, and modes are reordered by the commutator formula $[a_m, b_n] = \sum_j \binom{m + \Delta_a - 1}{j} (a_{(j)} b)_{m+n}$. `V.verma_module(h)` is the case of a one-dimensional $U$.

The Virasoro Verma module $M(c, h)$, with $c$ and $h$ symbolic:

In [13]:
Kch.<c, h> = Frac(QQ['c', 'h'])
Eh.<T> = NormallyOrderedExpressions(Kch)
VirH = UniversalVertexAlgebra(Kch, Eh, {('T', 'T'): [T.derivative(), 2*T, 0, c/2]}, conformal_vector=T)
T = VirH.gen('T')
Mh = VirH.verma_module(h)
v = Mh.highest_weight_vector()
w = T.mode(T.mode(v, -1), -1)
print(Mh, '\n', w)
print(T.mode(w, 1), '|', T.mode(w, 2), '|', T.mode(w, 0) == (h + 2)*w)
print([Mh.weight_space(h + d).dimension() for d in range(8)])
print(Mh.weight_space(h + 2).basis())

Verma module of highest weight {'T': h} over Universal vertex algebra on generators ['T'] over Fraction Field of Multivariate Polynomial Ring in c, h over Rational Field 
 T_{-1} T_{-1} v
(4*h+2)*T_{-1} v | 6*h*v | True
[1, 1, 2, 3, 5, 7, 11, 15]
[T_{-2} v, T_{-1} T_{-1} v]


Singular vectors at depth 2: a vector of weight $h + 2$ is singular iff $L_1$ and $L_2$ kill it, so one exists iff the matrix of $(L_1, L_2)$ from the depth-2 space to the depth-1 and depth-0 spaces is singular. Its determinant is $-(16h^2 + 2(c - 5)h + c)$, which vanishes iff $c = 2h(5 - 8h)/(2h + 1)$ (the remaining factor $h$ of the Kac determinant belongs to the singular vector $L_{-1} v$ at depth 1, which exists iff $h = 0$):

In [14]:
S2 = Mh.weight_space(h + 2).basis()                      # L_{-2} v, L_{-1}^2 v
k1 = Mh.weight_space(h + 1).basis_keys()[0]              # L_{-1} v
k0 = Mh.weight_space(h).basis_keys()[0]                  # v
mat = matrix(Kch, [[T.mode(s, 1).monomial_coefficients().get(k1, 0),
                    T.mode(s, 2).monomial_coefficients().get(k0, 0)] for s in S2])
print(S2); print(mat)
print(factor(mat.det()))

[T_{-2} v, T_{-1} T_{-1} v]
[          3 1/2*c + 4*h]
[    4*h + 2         6*h]
(-1) * (2*c*h + 16*h^2 + c - 10*h)


At $c = h = 1/2$ (the Ising model) the singular vector is $(L_{-2} - \frac34 L_{-1}^2)v$; at $h = 0$ it is $L_{-1} v$:

In [15]:
Eq.<Ti> = NormallyOrderedExpressions(QQ)
Ising = UniversalVertexAlgebra(QQ, Eq, {('Ti', 'Ti'): [Ti.derivative(), 2*Ti, 0, 1/4]}, conformal_vector=Ti)
Ti = Ising.gen('Ti')
M12 = Ising.verma_module(1/2)
v = M12.highest_weight_vector()
s = Ti.mode(v, -2) - 3/4 * Ti.mode(Ti.mode(v, -1), -1)
print(s, '|', Ti.mode(s, 1), Ti.mode(s, 2))
M0 = Ising.verma_module(0)
print(Ti.mode(Ti.mode(M0.highest_weight_vector(), -1), 1))
print(M12.character(5))

Ti_{-2} v - 3/4*Ti_{-1} Ti_{-1} v | 0 0
0
q^(1/2) + q^(3/2) + 2*q^(5/2) + 3*q^(7/2) + 5*q^(9/2) + O(q^5)


Affine $\mathfrak{sl}_2$ at level $k$: the Weyl module of the doublet has top weight $3/(4(k+2))$ (the Sugawara vector on the doublet). Its vector $E_{-1} u_0$ satisfies $F_1 E_{-1} u_0 = (k - 1) u_0$, so it is singular exactly at level 1 (where the doublet is integrable):

In [16]:
Kk.<k> = Frac(QQ['k'])
Vk = UniversalAffineVertexAlgebra(Kk, ['A', 1], level=k, names=['E', 'H', 'F'], conformal_vector='sugawara')
W = Vk.induced_module(Vk.zhu_algebra().module(doublet))
print(W.top_weight(), [W.weight_space(W.top_weight() + d).dimension() for d in range(4)])
Ek, Hk, Fk = Vk.gens()
u0 = W.highest_weight_vector()
x = Ek.mode(u0, -1)
print(x, '|', Fk.mode(x, 1), '|', Hk.mode(x, 1), Ek.mode(x, 1))

3/4/(k + 2) [2, 6, 18, 44]
E_{-1} u[0] | (k-1)*u[0] | 0 0


On the top level the zero modes $o(a) = a_{(\Delta_a - 1)}$ act by the matrices of $U$ of the images $[a]$ in the Zhu algebra (Zhu's theorem); here for every basis vector of $V^k(\mathfrak{sl}_2)$ of weight 2:

In [17]:
Ak = Vk.zhu_algebra()
U = W.top_level()
ok = True
for key in Vk.weight_space(2).basis_keys():
    a = Vk.monomial(key)
    m = U.representation_matrix(Ak(a))
    for j in range(2):
        img = a.nproduct(W.monomial(((), j)), 1)
        ok = ok and dict(img.monomial_coefficients()) == {((), r): m[r, j] for r in range(2) if m[r, j]}
print(ok, len(Vk.weight_space(2).basis_keys()))

True 9


The trivial top: $V^k(\mathfrak{sl}_2)$ itself is the induced module of the trivial module of $U(\mathfrak{sl}_2)$, by $a \mapsto a_{(-1)} v$:

In [18]:
Mt = Vk.induced_module(Ak.module({'E': 0, 'H': 0, 'F': 0}))
vt = Mt.highest_weight_vector()
print([Mt.weight_space(d).dimension() for d in range(4)], [Vk.weight_space(d).dimension() for d in range(4)])
a, bb = Ek, Fk.no(Hk)
print(all(a.nproduct(bb, n).nproduct(vt, -1) == a.nproduct(bb.no(vt), n) for n in range(-2, 3)))

[1, 3, 9, 22] [1, 3, 9, 22]
True


Non-linear OPEs: the $W_3$ algebra. Here the reordering of modes involves the composite field $\Lambda = {:}LL{:} - \frac3{10}\partial^2 L$, and that the result is a module rests on the consistency of the rewriting, i.e. on the Jacobi identity of $V$. With Zamolodchikov's $\beta = 16/(22 + 5c)$ the commutator formula holds on all vectors of depth $\le 2$; with a wrong $\beta$ it fails. (This $W_3$ is given by conformal weights, without a conformal vector, so the module is graded by depth, top weight $0$; $L_0$ acts on the top by $c + 1$.)

In [19]:
K.<c> = Frac(QQ['c'])                                    # c was rebound to Frac(QQ['c', 'h']) above
E3.<L3, W3> = NormallyOrderedExpressions(K)
def w3(beta):
    Lam = L3*L3 - 3/10*L3.derivative(2)
    return UniversalVertexAlgebra(K, E3, {('L3', 'L3'): [L3.derivative(), 2*L3, 0, c/2],
        ('L3', 'W3'): [W3.derivative(), 3*W3],
        ('W3', 'W3'): [beta*Lam.derivative() + L3.derivative(3)/15, 2*beta*Lam + 3/10*L3.derivative(2),
                       L3.derivative(), 2*L3, 0, c/3]}, degrees={'L3': 2, 'W3': 3})

def failures(V):
    M = V.verma_module({'L3': c + 1, 'W3': 1/c})
    Lg, Wg = V.gens()
    bad = total = 0
    for x, y in [(Wg, Wg), (Wg, Lg)]:
        xy = x.ope(y)
        for m in [M.monomial(kk) for d in (0, 1, 2) for kk in M._depth_keys(d)]:
            for q in range(-1, 3):
                for p in range(4):
                    lhs = x.nproduct(y.nproduct(m, q), p) - y.nproduct(x.nproduct(m, p), q)
                    rhs = sum((binomial(p, j) * xy[j].nproduct(m, p + q - j)
                               for j in range(min(p + 1, len(xy)))), M.zero())
                    total += 1
                    bad += lhs != rhs
    return bad, total

print('Zamolodchikov:', failures(w3(16/(22 + 5*c))), '  wrong beta:', failures(w3(1)))
MW = w3(16/(22 + 5*c)).verma_module({'L3': c + 1, 'W3': 1/c})
print(MW.top_weight(), [MW.weight_space(d).dimension() for d in range(6)])   # weights only, no T: top weight 0
TestSuite(MW).run()

Zamolodchikov: (0, 256)   wrong beta: (9, 256)
0 [1, 2, 5, 10, 20, 36]


## 5. Submodules, quotients and simple modules

`M.submodule(vectors)` is the submodule *generated* by the vectors (all $a_{(n)} v$, repeated), `M / N` the quotient, `M.maximal_submodule()` the maximal proper submodule and `M.simple_quotient()` = `M / M.maximal_submodule()`, Zhu's $L(U)$ for an induced module. Weight spaces are computed exactly: a submodule is spanned by raising words in the modes of the strong generators applied to the closure of its generators under the non-raising modes; the maximal submodule (the largest graded submodule meeting the top level trivially) is computed depth by depth as the vectors that every lowering mode sends into it. Submodules are modules themselves, with canonical bases of their weight spaces (reduced echelon forms, leading monomials as pivots); quotient vectors are normal forms.

The Ising model: the three simple quotients $L(1/2, h)$ of Verma modules and their characters, $q^{1/16}\prod_n (1 + q^n)$ for $h = 1/16$:

In [20]:
for h in (0, 1/2, 1/16):
    S = Ising.verma_module(h).simple_quotient()
    print(h, [S.weight_space(h + d).dimension() for d in range(11)])
print(S, '\n', S.character(6))

0 [1, 0, 1, 1, 2, 2, 3, 3, 5, 5, 7]
1/2 [1, 1, 1, 1, 2, 2, 3, 4, 5, 6, 8]
1/16 [1, 1, 1, 2, 2, 3, 4, 5, 6, 8, 10]
Simple quotient of Verma module of highest weight {'Ti': 1/16} over Universal vertex algebra on generators ['Ti'] over Rational Field 
 q^(1/16) + q^(17/16) + q^(33/16) + 2*q^(49/16) + 2*q^(65/16) + 3*q^(81/16) + O(q^6)


In $M(1/2, 1/2)$ the singular vector $s = (L_{-2} - \frac34 L_{-1}^2) v$ generates a copy of $M(1/2, 5/2)$ (the dimensions are those of a Verma module two levels up); `express` writes a vector of the submodule as a descendant of $s$ (in the $(n)$-products, here $T_{(0)} = L_{-1}$, $T_{(-2)} = L_{-3}$). The maximal submodule also contains the singular vector of depth 3, and is the sum of the two submodules they generate:

In [21]:
M = Ising.verma_module(1/2); v = M.highest_weight_vector()
s = Ti.mode(v, -2) - 3/4 * Ti.mode(Ti.mode(v, -1), -1)
N = M.submodule([s], names=['s'])
print(N, '\n', [N.weight_space(1/2 + d).dimension() for d in range(9)])
x = Ti.mode(Ti.mode(s, -1), -1) + 2 * Ti.mode(s, -3)
print(x in N, v in N, '|', N.express(x))
J = M.maximal_submodule()
print(J.weight_space(1/2 + 2).basis(), [J.weight_space(1/2 + d).dimension() for d in range(9)])
Y = M.weight_space(1/2 + 3).basis()                     # the singular vectors of depth 3
keys = sorted({k for y in Y for m in (1, 2) for k in Ti.mode(y, m).monomial_coefficients()})
A = matrix(QQ, [[Ti.mode(y, m).monomial_coefficients().get(k, 0) for m in (1, 2) for k in keys] for y in Y])
s3 = sum(cf * y for cf, y in zip(A.left_kernel().basis()[0], Y)); print(s3)
N3 = M.submodule([s3], names=['s3'])
print([(N + N3).weight_space(1/2 + d).dimension() for d in range(9)], N3 <= N, N <= N + N3)

Submodule of Verma module of highest weight {'Ti': 1/2} over Universal vertex algebra on generators ['Ti'] over Rational Field generated by [Ti_{-2} v - 3/4*Ti_{-1} Ti_{-1} v]

 
 [0, 0, 1, 1, 2, 3, 5, 7, 11]
True False | Ti_(0) Ti_(0) s + 2*Ti_(-2) s
[-4/3*Ti_{-2} v + Ti_{-1} Ti_{-1} v] [0, 0, 1, 2, 3, 5, 8, 11, 17]
Ti_{-3} v - 4*Ti_{-2} Ti_{-1} v + 4/3*Ti_{-1} Ti_{-1} Ti_{-1} v
[0, 0, 1, 2, 3, 5, 8, 11, 17] False True


Lee–Yang: the simple quotient of the adjoint module of the universal Virasoro algebra at $c = -22/5$ is the quotient by the maximal ideal, generated by $\Lambda$; the Verma module $M(-22/5, -1/5)$ has the other Rogers–Ramanujan character, and in its simple quotient $L_{-1}^2 v = \frac25 L_{-2} v$:

In [22]:
LYa = VirLY.adjoint_module().simple_quotient()
print([LYa.weight_space(w).dimension() for w in range(11)])
print([VirLY.quotient([Lq.no(Lq) - 3/10*Lq.derivative(2)]).weight_space(w).dimension() for w in range(11)])
S = VirLY.verma_module(-1/5).simple_quotient(); v = S.highest_weight_vector()
print([S.weight_space(-1/5 + d).dimension() for d in range(11)], Lq.mode(Lq.mode(v, -1), -1))

[1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4]
[1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4]
[1, 1, 1, 1, 2, 2, 3, 3, 4, 5, 6] 2/5*Lq_{-2} v


Affine $\mathfrak{sl}_2$ at level 1: the simple quotients of the Weyl modules of the trivial module and of the doublet are the two lattice modules of $A_1$ (section 2); at level 2 the vacuum gives $L_2(\mathfrak{sl}_2)$:

In [23]:
for U, LM in zip((AA.module({'E': 0, 'H': 0, 'F': 0}), AA.module(doublet)), V.simple_modules()):
    L = VA.induced_module(U).simple_quotient()
    h = L.top_weight()
    print(h, [L.weight_space(h + d).dimension() for d in range(6)], [LM.weight_space(h + d).dimension() for d in range(6)])
V2 = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=2, names=['E', 'H', 'F'], conformal_vector='sugawara')
L2 = V2.induced_module(V2.zhu_algebra().module({'E': 0, 'H': 0, 'F': 0})).simple_quotient()
print([L2.weight_space(d).dimension() for d in range(6)])

0

 [1, 3, 4, 7, 13, 19] [1, 3, 4, 7, 13, 19]


1/4 [2, 2, 6, 8, 14, 20] [2, 2, 6, 8, 14, 20]


[1, 3, 9, 15, 30, 54]


Direct sums, and a quotient of one by a diagonal submodule; lattice modules are simple:

In [24]:
Ma, Mb = Ising.verma_module(0), Ising.verma_module(1/16)
D = Ma.direct_sum(Mb)
x = D.embed(0, Ma.highest_weight_vector()) + D.embed(1, Mb.highest_weight_vector())
print(Ti.mode(x, -1), '|', D.character(3))
DD = Ma.direct_sum(Ma)
diag = DD.submodule([DD.embed(0, Ma.highest_weight_vector()) + DD.embed(1, Ma.highest_weight_vector())])
print((DD / diag).character(5) == Ma.character(5))
print(M1.simple_quotient() is M1, M1.maximal_submodule().weight_space(1/4).dimension())
TestSuite(DD / diag).run(); TestSuite(J).run()

[0] Ti_{-1} v + [1] Ti_{-1} v | 1 + q^(1/16) + q + q^(17/16) + 2*q^2 + 2*q^(33/16) + O(q^3)
True
True 0


## 6. Gradings by abelian groups

If $V$ is graded by an abelian group $A$, a module is graded by a group $B$ containing $A$ (an embedding $\iota: A \to B$, `M.horizontal_grading_embedding()`), with $a_{(n)} m$ of grade $\iota(\mathrm{grade}\, a) + \mathrm{grade}\, m$; the grades of an indecomposable module lie in one coset of $\iota(A)$, so $B$ is in general larger than $A$. `M.horizontal_grading_group()`, `M.horizontal_grade(v)`, and `weight_space` / `character` take a `horizontal_grade` in $B$.

Lattice modules: with the lattice grading of $V_L$ the grades lie in the dual lattice $L^\circ$, $e^p$ of grade $p$ (for another grading of $V_L$, in the pushout of $A \leftarrow L \rightarrow L^\circ$); all simple modules share $B$, so direct sums are graded:

In [25]:
Vg = V.with_horizontal_grading('lattice')
N0, N1 = Vg.simple_modules()
print(N1.horizontal_grading_group(), '|', N1.horizontal_grade(N1.highest_weight_vector()), N1.horizontal_grading_embedding()(1))
print([N1.weight_space(1/4 + d, horizontal_grade=(1/2,)).dimension() for d in range(6)])   # partitions
print(N0.direct_sum(N1).character(3, horizontal_grade=(-1/2,)))
A2g = A2.with_horizontal_grading([1, 2], group=Zmod(3))
B3 = A2g.simple_modules()[1].horizontal_grading_group(); print(B3, B3.cardinality())

Free module of degree 1 and rank 1 over Integer Ring
Echelon basis matrix:
[1/2] | (-1/2) (1)
[1, 1, 2, 3, 5, 7]
q^(1/4) + q^(5/4) + 2*q^(9/4) + O(q^3)
Finitely generated module V/W over Integer Ring with invariants (9) 9


Affine vertex algebras are graded by the root lattice $Q$; their modules by the weights $P \otimes R$, $R$ the coefficient ring (fundamental weight coordinates), which contains $Q$ by the Cartan matrix. The grades of a module over the Zhu algebra are by default the eigenvalues of the zero modes of the Cartan currents (or given with `horizontal_grades=`); an induced module grades its PBW monomials accordingly. The Weyl module of the doublet over $\mathbb Q(k)$, and at level 1 the simple quotient, graded like the lattice module $V_{1/2 + A_1}$ ($J_{(0)} e^p = 2p\, e^p$):

In [26]:
U = W.top_level(); print(U.horizontal_grades(), U.horizontal_grading_group())
h = W.top_weight()
print([[W.weight_space(h + d, horizontal_grade=(m,)).dimension() for m in range(-5, 6, 2)] for d in range(4)])
L1 = VA.induced_module(AA.module(doublet)).simple_quotient()
print([[L1.weight_space(1/4 + d, horizontal_grade=(m,)).dimension() for m in range(-5, 6, 2)] for d in range(4)])
print([[N1.weight_space(1/4 + d, horizontal_grade=(m/2,)).dimension() for m in range(-5, 6, 2)] for d in range(4)])

[(1), (-1)] Vector space of dimension 1 over Fraction Field of Univariate Polynomial Ring in k over Rational Field
[[0, 0, 1, 1, 0, 0], [0, 1, 2, 2, 1, 0], [1, 3, 5, 5, 3, 1], [3, 7, 11, 11, 7, 3]]
[[0, 0, 1, 1, 0, 0], [0, 0, 1, 1, 0, 0], [0, 1, 2, 2, 1, 0], [0, 1, 3, 3, 1, 0]]
[[0, 0, 1, 1, 0, 0], [0, 0, 1, 1, 0, 0], [0, 1, 2, 2, 1, 0], [0, 1, 3, 3, 1, 0]]


Grades outside $A$: the $N = 2$ algebra graded by the $J$-charge ($G^\pm$ of charge $\pm 1$) and its Verma module of charge $1/3$, graded by $\mathbb Q \supset \mathbb Z$:

In [27]:
X2.<Ln, Jn, Gp, Gm> = NormallyOrderedExpressions(QQ, parities={'Gp': 1, 'Gm': 1})
N2 = UniversalVertexAlgebra(QQ, X2, {
    ('Ln', 'Ln'): [Ln.derivative(), 2*Ln, 0, 1/2], ('Ln', 'Jn'): [Jn.derivative(), Jn],
    ('Ln', 'Gp'): [Gp.derivative(), 3/2*Gp], ('Ln', 'Gm'): [Gm.derivative(), 3/2*Gm],
    ('Jn', 'Jn'): [0, 1/3], ('Jn', 'Gp'): [Gp], ('Jn', 'Gm'): [-Gm],
    ('Gp', 'Gm'): [Ln + Jn.derivative()/2, Jn, 1/3]}, conformal_vector=Ln, horizontal_grading={'Gp': 1, 'Gm': -1})
MN = N2.induced_module(N2.zhu_algebra().module({'Ln': 1/6, 'Jn': 1/3}, horizontal_grades=[(1/3,)]))
print(MN.horizontal_grading_group(), MN.horizontal_grade(MN.highest_weight_vector()))
print([MN.weight_space(1/6 + d/2, horizontal_grade=(1/3 - 1,)).dimension() for d in range(9)])
print(MN.character(3, horizontal_grade=(1/3,)))

Vector space of dimension 1 over Rational Field (1/3)
[0, 1, 0, 3, 0, 9, 0, 22, 0]
q^(1/6) + 3*q^(7/6) + 9*q^(13/6) + O(q^3)


## 7. Infinite top levels: affine Verma modules and admissible levels

Zhu algebras inherit the grading of $V$ (`A.horizontal_grading_group()`, `x.horizontal_grade()`). A functional on the grading group (the height for affine algebras) splits the generators of $A(V)$ into raising, Cartan and lowering ones; `A.verma_module(λ)` is the (infinite-dimensional, graded) Verma module of $A(V)$, here $M_{\mathfrak{sl}_2}(\lambda)$ of the admissible weight $\lambda = -1/2$ at level $k = -1/2$:

In [28]:
Va = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=-1/2, names=['E', 'H', 'F'], conformal_vector='sugawara')
Aa = Va.zhu_algebra(); Ez, Hz, Fz = Aa.gens()
print(Aa.horizontal_grading_group(), (Ez*Fz).horizontal_grade(), Aa.homogeneous_components(Ez*Ez + Ez))
Ua = Aa.verma_module({'H': -1/2}); v = Ua.highest_weight_vector()
x = Fz * (Fz * v); print(Ua, '\n', x, '|', Ez * x, '|', Ua.horizontal_grade(x), '|', Ua._scalar(Aa(Va.conformal_vector())))

Ambient free module of rank 1 over the principal ideal domain Integer Ring (0) {(2): [E]^2, (1): [E]}
Verma module of highest weight {'H': -1/2} over Zhu algebra of Universal affine vertex algebra of type ['A', 1] at level -1/2 over Rational Field 
 [F]^2 v | -3*[F] v | (-9/2) | -1/8


Inducing gives the affine Verma module $M(\lambda)$: its conformal weight spaces are infinite dimensional, those of fixed $\mathfrak{sl}_2$-weight finite. Submodules and the maximal submodule are computed piece by piece in the twisted degree depth $+\,\mu(\text{grade})$ ($\mu$ a small multiple of minus the height: the highest weight vector is the only vector of degree 0, every piece is finite) and the grade, so only the pieces a question needs are computed, each a small linear algebra problem. The simple quotient $L(-1/2)$ and $L(-3/2)$: they are the two halves of the Ramond sector of the $\beta\gamma$ system (integer modes, $\beta_0 v = 0$), counted here by depth and charge $\#\beta - \#\gamma$:

In [29]:
def fock(d, c):
    return sum(1 for db in range(d + 1) for pb in Partitions(db) for pg in Partitions(d - db)
               if len(pb) - len(pg) - c >= 0)

for lam in (-1/2, -3/2):
    Ma = Va.induced_module(Aa.verma_module({'H': lam}))
    La = Ma.simple_quotient(); h = Ma.top_weight()
    ours = [[La.weight_space(h + d, horizontal_grade=(lam + m,)).dimension() for m in range(-6, 3, 2)] for d in range(4)]
    want = [[fock(d, lam + m + 1/2) for m in range(-6, 3, 2)] for d in range(4)]
    print(lam, h, ours == want, ours)
try:
    Ma.weight_space(h)
except ValueError as err:
    print('ValueError:', err)

-1/2 -1/8 True [[1, 1, 1, 1, 0], [2, 2, 2, 1, 0], [5, 5, 5, 3, 1], [10, 10, 9, 6, 2]]


-3/2

 -1/8 True [[1, 1, 1, 1, 0], [2, 2, 2, 2, 1], [5, 5, 5, 4, 2], [10, 10, 10, 8, 4]]
ValueError: the conformal weight spaces of Induced module of Verma module of highest weight {'H': -3/2} over Zhu algebra of Universal affine vertex algebra of type ['A', 1] at level -1/2 over Rational Field over Universal affine vertex algebra of type ['A', 1] at level -1/2 over Rational Field are infinite dimensional: give a grade


$\mathfrak{sl}_3$ at the admissible level $-3/2$ and $\lambda = (-\frac12, -\frac12)$ (fundamental weight coordinates): of the reflections only $\theta$ gives a singular vector of the top ($\langle \lambda + \rho, \theta^\vee \rangle = 1$), so $L(\lambda)$ loses one vector at $\lambda - \theta$:

In [30]:
Vs = UniversalAffineVertexAlgebra(QQ, ['A', 2], level=-3/2, conformal_vector='sugawara')
Ms = Vs.induced_module(Vs.zhu_algebra().verma_module({'alphacheck[1]': -1/2, 'alphacheck[2]': -1/2}))
Ls = Ms.simple_quotient(); hs = Ms.top_weight()
C = matrix(ZZ, [[2, -1], [-1, 2]]); lam = vector(QQ, [-1/2, -1/2])
for d in range(3):
    print(d, [(a, b, Ms.weight_space(hs + d, horizontal_grade=tuple(lam - C*vector([a, b]))).dimension(),
               Ls.weight_space(hs + d, horizontal_grade=tuple(lam - C*vector([a, b]))).dimension())
              for a in range(3) for b in range(3)])

0 [(0, 0, 1, 1), (0, 1, 1, 1), (0, 2, 1, 1), (1, 0, 1, 1), (1, 1, 2, 1), (1, 2, 2, 1), (2, 0, 1, 1), (2, 1, 2, 1), (2, 2, 3, 1)]


1 [(0, 0, 6, 3), (0, 1, 8, 3), (0, 2, 8, 3), (1, 0, 8, 3), (1, 1, 14, 4), (1, 2, 16, 4), (2, 0, 8, 3), (2, 1, 16, 4), (2, 2, 22, 4)]


2 [(0, 0, 33, 9), (0, 1, 44, 10), (0, 2, 47, 10), (1, 0, 44, 10), (1, 1, 72, 13), (1, 2, 85, 13), (2, 0, 47, 10), (2, 1, 85, 13), (2, 2, 116, 14)]


## 8. Modules over quotients

For a quotient $Q = V/I$ of a universal vertex algebra (also a presentation $F/R$ of a subalgebra by its strong generators, `S.presentation()`) and a finite-dimensional module $U$ over a Zhu algebra of $Q$ (`Q.zhu_algebra_up_to(W)`), `Q.induced_module_up_to(U, max_weight, max_depth)` is Dong–Li–Mason's universal module $\bar M_Q(U)$ with top level $U$: every positive energy $Q$-module generated by $U$, simple or logarithmic, is a quotient of it. It is $\bar M_V(U)$ modulo the subspace generated by the $r_{(n)} u$, $r \in I$, $u \in U$ (and by the defects of the commutator formula of the generators, which occur when $V$ does not satisfy the Jacobi identity), computed in the depths $\le$ `max_depth` from the relations of weight $\le$ `max_weight`. It maps onto $\bar M_Q(U)$ in these depths and equals it once it no longer changes as `max_weight` and `max_depth` grow.

The Lee–Yang model is rational, so $\bar M_Q(U)$ is the simple module $L(U)$: from its two tops, the two Rogers–Ramanujan characters of section 5, now as modules over the quotient. A top that is not a module over the Zhu algebra of $Q$ is reached by the relations and refused:

In [31]:
QLY = VirLY.quotient([Lq.no(Lq) - 3/10*Lq.derivative(2)])
ZLY = QLY.zhu_algebra_up_to(4)
for h in (0, -1/5):
    M = QLY.induced_module_up_to(ZLY.module({'Lq': h}), 4, 10)
    print(h, [M.weight_space(h + d).dimension() for d in range(11)])
try:
    QLY.induced_module_up_to(VirLY.zhu_algebra().module({'Lq': 1/3}), 4, 3)
except ValueError as err:
    print('ValueError:', str(err)[:80], '...')

0 [1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4]
-1/5 [1, 1, 1, 1, 2, 2, 3, 3, 4, 5, 6]
ValueError: the relations of weight <= 4 reach the top level: Module of dimension 1 over Zhu ...


The triplet algebra $\mathcal W(2)$, a presentation of a subalgebra of symplectic fermions with logarithmic modules, is worked out in the notebook `triplet_modules.ipynb`: its Zhu algebra and simple modules, the module $\mathcal W(0) \to 2\mathcal W(1)$ and the projective cover of its vacuum module.

## 9. Not there yet

* relaxed modules (dense modules of the Zhu algebra as tops: infinite in both directions, so their submodules need more than the twisted degree); Verma modules of quotients of Zhu algebras;

* infinite top levels over quotients $V/I$; quotients of submodules;
* tensor products of modules, restriction to subalgebras and cosets, contragredient modules;
* Ramond-twisted induced modules, generators of weight 0;
* the C++ engines for module actions (lattice modules run in Python).